# Phase 2. Search

**Md Forhadul Islam**. Review of benchmarks for coding agents and agentic software engineering (2022 to 2026).

Two sources, open access only: **OpenAlex** (five query blocks in titles and abstracts, 1 November 2022 to 9 November 2026)
and **Google Scholar** (five queries since 2022, top 50 results each).

The searches are run **again today, live**, and saved in `search/live_rerun_<date>/`. The review itself continues with the
results saved on 9 and 10 November 2026 (in `data/`), because the screening decisions belong to those records. The last
section compares today's results with 9 November.

In [ ]:
from setup import *
assert Path.cwd() == ROOT, "the notebook must work inside the Report folder"
print("Report folder:", ROOT)

## 2.1 OpenAlex search (live)

In [ ]:
if OFFLINE:
    print("offline test run: this live internet step is skipped")
else:
    import urllib.request, urllib.parse
    LIVE = ROOT / 'search' / f'live_rerun_{datetime.date.today().isoformat()}'; os.makedirs(LIVE, exist_ok=True)
    API = 'https://api.openalex.org/works'
    FIELDS = 'id,doi,title,publication_year,publication_date,type,cited_by_count,primary_location,best_oa_location,open_access,authorships,abstract_inverted_index,referenced_works'
    QUERIES = {'A': '("coding agent" OR "coding agents" OR "software engineering agent" OR "code agent") AND (benchmark OR dataset)',
               'B': '("SWE-bench" OR "repository-level code" OR "issue resolution") AND (benchmark OR dataset)',
               'C': '("agentic" AND "software engineering") AND (benchmark OR evaluation)',
               'D': '("data contamination" OR "benchmark contamination") AND (code OR "software engineering")',
               'E': '("coding agents" OR "code generation") AND benchmark AND (survey OR "systematic review")'}
    def get(url):
        for i in range(4):
            try:
                with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'review-reproduction (mailto not given)'}), timeout=90) as r: return json.loads(r.read())
            except Exception as e:
                print('retry', i + 1, type(e).__name__); time.sleep(5 * (i + 1))
        raise RuntimeError('OpenAlex not reachable')
    def pages(filter_text, select=FIELDS):
        page, out = 1, []
        while True:
            j = get(f'{API}?filter={filter_text}&per-page=200&select={select}&page={page}')
            out += j['results']
            if page * 200 >= j['meta']['count'] or page >= 50: return out, j['meta']['count']
            page += 1; time.sleep(0.2)
    def abstract(inv):
        if not inv: return ''
        w = {}
        for word, pos in inv.items():
            for p in pos: w[p] = word
        return ' '.join(w[i] for i in sorted(w))
    records, log = {}, []
    for k, q in QUERIES.items():
        works, total = pages('title_and_abstract.search:' + urllib.parse.quote(q) + ',from_publication_date:2022-11-01,to_publication_date:2026-11-09')
        for w in works:
            r = records.setdefault(w['id'], {'id': w['id'], 'doi': w.get('doi'), 'title': w.get('title'), 'year': w.get('publication_year'), 'queries': []})
            r['queries'].append(k)
        log.append({'k': k, 'query': q, 'total_hits': total, 'fetched': len(works), 'run_utc': datetime.datetime.utcnow().isoformat()})
        print(k, total, 'hits')
    json.dump({'retrieved_utc': datetime.datetime.utcnow().isoformat(), 'log': log, 'records': list(records.values())}, open(LIVE / 'openalex_live.json', 'w'), indent=0)
    print('OpenAlex today:', len(records), 'records (one per work)')

## 2.2 Google Scholar search (live)

Scholar has no export or API. The cell reads the first five result pages of each query with a pause between pages, and stops if Scholar asks for a CAPTCHA (then the same JavaScript is run in the browser: `scripts/browser_js/02_google_scholar.js`).

In [ ]:
if OFFLINE:
    print("offline test run: this live internet step is skipped")
else:
    from html.parser import HTMLParser
    GQ = {'G1': '("coding agents" OR "software engineering agents" OR "code agents") benchmark', 'G2': '"SWE-bench" benchmark OR dataset "agents"',
          'G3': '"repository-level" code benchmark agents evaluation', 'G4': 'benchmark contamination "coding agents" OR "code generation" OR "software engineering"',
          'G5': '("coding agents" OR "agentic software engineering") survey OR "systematic review" benchmarks'}
    class Titles(HTMLParser):
        def __init__(self): super().__init__(); self.t = []; self.on = False; self.cur = ''
        def handle_starttag(self, tag, a):
            if tag == 'h3' and ('class', 'gs_rt') in a: self.on = True; self.cur = ''
        def handle_endtag(self, tag):
            if tag == 'h3' and self.on: self.on = False; self.t.append(self.cur.strip())
        def handle_data(self, d):
            if self.on: self.cur += d
    gs, gslog, blocked = [], [], False
    for k, q in GQ.items():
        if blocked: break
        for p in range(5):
            url = 'https://scholar.google.com/scholar?hl=en&as_sdt=0,5&as_ylo=2022&q=' + urllib.parse.quote(q) + f'&start={p * 10}'
            html = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}), timeout=60).read().decode('utf-8', 'ignore')
            if re.search('unusual traffic|captcha|not a robot', html, re.I): blocked = True; print('Scholar asked for a CAPTCHA; stopping'); break
            tp = Titles(); tp.feed(html); gs += [{'q': k, 'title': t} for t in tp.t]
            if p == 0:
                m = re.search(r'About ([\d,]+) results', html); gslog.append({'k': k, 'query': q, 'scholar_total_text': m.group(0) if m else ''})
            time.sleep(4)
        print(k, gslog[-1]['scholar_total_text'] if gslog else '', '|', sum(1 for r in gs if r['q'] == k), 'results read')
    json.dump({'log': gslog, 'records': gs, 'blocked': blocked}, open(LIVE / 'scholar_live.json', 'w'), indent=0)
    if not gs and not blocked:
        print('Scholar sent pages that Python could not read. As on 9 November, the same search is then run in Chrome with '
              'scripts/browser_js/02_google_scholar.js and saved as scholar_live_browser.json in this folder; section 2.4 uses that file.')

## 2.3 The search results of 9 November 2026 (used by the review)

In [ ]:
OA = J('data/openalex_raw_2026-11-09.json'); GS = J('data/scholar_raw_2026-11-09.json')
for l in OA['log']: print(l['k'], l.get('total_hits'), 'hits |', l['query'])
print('OpenAlex records:', len(OA['records'])); expect('OpenAlex records on 9 November', len(OA['records']), 3364)
for l in GS['log']: print(l['k'], l['scholar_total_text'], '| read', l['fetched'])
expect('Google Scholar results on 9 November', len(GS['records']), 250)

## 2.4 Today against 9 November

In [ ]:
if OFFLINE:
    print("offline test run: this live internet step is skipped")
else:
    norm = lambda t: re.sub(r'[^a-z0-9]+', ' ', re.sub(r'^(\s*\[[A-Z]+\]\s*)+', '', t or '').lower()).strip()
    live = J(str((LIVE / 'openalex_live.json').relative_to(ROOT)))
    old_ids = {r['id'] for r in OA['records']}; new_ids = {r['id'] for r in live['records']}
    print(f"OpenAlex: today {len(new_ids)}, 9 November {len(old_ids)}, found both times {len(old_ids & new_ids)}, only today {len(new_ids - old_ids)}, only on 9 November {len(old_ids - new_ids)}")
    check('OpenAlex today finds at least 95 percent of the 9 November records', len(old_ids & new_ids) >= 0.95 * len(old_ids), f'{100 * len(old_ids & new_ids) / len(old_ids):.1f} percent')
    bf = LIVE / 'scholar_live_browser.json'
    gl = J(str((bf if bf.exists() else LIVE / 'scholar_live.json').relative_to(ROOT)))
    print('Scholar results read from', 'the browser run' if bf.exists() else 'the Python run')
    a = {norm(r['title']) for r in GS['records']}; b = {norm(r['title']) for r in gl['records']}
    print(f'Google Scholar: today {len(b)} titles, 9 November {len(a)}, found both times {len(a & b)}' + (' (stopped by a CAPTCHA)' if gl.get('blocked') else ''))
    json.dump({'openalex': {'today': len(new_ids), 'nov9': len(old_ids), 'both': len(old_ids & new_ids)}, 'scholar': {'today': len(b), 'nov9': len(a), 'both': len(a & b), 'blocked': gl.get('blocked')}},
              open(LIVE / 'comparison_with_9_november.json', 'w'), indent=1)

## 2.5 The browser versions of the searches (as run on 9 November 2026)

In [ ]:
os.makedirs(ROOT / 'scripts/browser_js', exist_ok=True)

In [ ]:
%%writefile scripts/browser_js/01_openalex_search.js
// Run on https://api.openalex.org. Queries A to E, title and abstract search, 2022-11-01 to 2026-11-09, page paging (no key: cursor paging needs a key).
const qs={A:'("coding agent" OR "coding agents" OR "software engineering agent" OR "code agent") AND (benchmark OR dataset)',
B:'("SWE-bench" OR "repository-level code" OR "issue resolution") AND (benchmark OR dataset)',
C:'("agentic" AND "software engineering") AND (benchmark OR evaluation)',
D:'("data contamination" OR "benchmark contamination") AND (code OR "software engineering")',
E:'("coding agents" OR "code generation") AND benchmark AND (survey OR "systematic review")'};
const sel='id,doi,title,publication_year,publication_date,type,cited_by_count,primary_location,best_oa_location,open_access,authorships,abstract_inverted_index,referenced_works';
function abs(ii){if(!ii)return '';const a=[];for(const[w,p]of Object.entries(ii))for(const i of p)a[i]=w;return a.join(' ');}
const R={},LOG=[];
for(const [k,q] of Object.entries(qs)){let page=1,total=null,n=0;
 while(true){const j=await (await fetch('https://api.openalex.org/works?filter=title_and_abstract.search:'+encodeURIComponent(q)+',from_publication_date:2022-11-01,to_publication_date:2026-11-09&per-page=200&select='+sel+'&page='+page)).json();
  total=j.meta.count;for(const w of j.results){const o=R[w.id]||(R[w.id]={id:w.id,doi:w.doi,title:w.title,year:w.publication_year,date:w.publication_date,type:w.type,cites:w.cited_by_count,venue:(w.primary_location&&w.primary_location.source&&w.primary_location.source.display_name)||'',oa:w.open_access&&w.open_access.is_oa,oa_url:(w.best_oa_location&&(w.best_oa_location.pdf_url||w.best_oa_location.landing_page_url))||'',authors:(w.authorships||[]).slice(0,6).map(a=>a.author.display_name).join('; '),abstract:abs(w.abstract_inverted_index),refs:(w.referenced_works||[]).join('|'),queries:[]});if(!o.queries.includes(k))o.queries.push(k);n++;}
  if(page*200>=total)break;page++;}
 LOG.push({k,query:q,total_hits:total,fetched:n,run_utc:new Date().toISOString()});}
// saved as openalex_raw_2026-11-09.json = {retrieved_utc, source, window, log: LOG, records: Object.values(R)}

In [ ]:
%%writefile scripts/browser_js/02_google_scholar.js
// Run on https://scholar.google.com. 5 queries, since 2022, first 5 result pages (50 results) each, 2.5 s pause between pages; stop on any CAPTCHA.
const qs={G1:'("coding agents" OR "software engineering agents" OR "code agents") benchmark',G2:'"SWE-bench" benchmark OR dataset "agents"',G3:'"repository-level" code benchmark agents evaluation',G4:'benchmark contamination "coding agents" OR "code generation" OR "software engineering"',G5:'("coding agents" OR "agentic software engineering") survey OR "systematic review" benchmarks'};
const out={log:[],recs:[]};const sleep=ms=>new Promise(r=>setTimeout(r,ms));
for(const [k,q] of Object.entries(qs)){let total=null,got=0;
 for(let p=0;p<5;p++){const html=await (await fetch('https://scholar.google.com/scholar?hl=en&as_sdt=0,5&as_ylo=2022&q='+encodeURIComponent(q)+'&start='+(p*10))).text();
  if(/unusual traffic|captcha|not a robot/i.test(html))throw 'blocked';
  const d=new DOMParser().parseFromString(html,'text/html');if(p===0)total=(d.querySelector('#gs_ab_md')||{}).innerText;
  d.querySelectorAll('.gs_r.gs_or').forEach(e=>{const a=e.querySelector('.gs_rt a');const cb=[...e.querySelectorAll('.gs_fl a')].find(x=>/Cited by/.test(x.innerText));const pdf=e.querySelector('.gs_or_ggsm a');
   out.recs.push({q:k,title:(e.querySelector('.gs_rt')||{}).innerText||'',url:a?a.href:'',meta:(e.querySelector('.gs_a')||{}).innerText||'',snippet:(e.querySelector('.gs_rs')||{}).innerText||'',cited_by:cb?parseInt(cb.innerText.replace(/\D/g,'')):0,pdf:pdf?pdf.href:''});got++;});
  await sleep(2500);}
 out.log.push({k,query:q,scholar_total_text:total,fetched:got,run_utc:new Date().toISOString()});}
// saved as scholar_raw_2026-11-09.json

In [ ]:
%%writefile scripts/browser_js/03_forward_snowball.js
// Forward snowballing: all works citing each seed, published from 2022-11-01 (OpenAlex filter cites:W...).
// Seeds round 1: SWE-bench W4387561453, RepoBench W4379548473, CrossCodeEval W7133187544, Multi-SWE-bench W4410313682, SWE-bench Multimodal W4403322180, SWE-bench Goes Live, LiveCodeBench, SWE-Lancer, Commit0 (ids resolved by title search). Then SWE-agent W7103754783 (resolved by DOI 10.48550/arXiv.2405.15793).
async function forward(seedId,label,R){let page=1,total=0;
 while(true){const j=await (await fetch('https://api.openalex.org/works?filter=cites:'+seedId+',from_publication_date:2022-11-01&per-page=200&select=id,doi,title,publication_year,publication_date,type,cited_by_count,primary_location,best_oa_location,open_access,authorships,abstract_inverted_index,referenced_works&page='+page)).json();
  total=j.meta.count;for(const w of j.results){/* same record mapping as 01, plus cites_seeds list */ (R[w.id]=R[w.id]||{id:w.id,title:w.title,cites_seeds:[]}).cites_seeds.push(label);}
  if(page*200>=total)break;page++;}
 return total;}
// saved as openalex_forward_snowball_round1_2026-11-09.json and openalex_forward_sweagent_and_seeds_2026-11-09.json

In [ ]:
%%writefile scripts/browser_js/04_backward_snowball.js
// Backward snowballing: works referenced by the 1,185 stage 1 Include or Maybe papers, published from 2022-11-01.
// Uses the OpenAlex filter cited_by:W1|W2|... in groups of 40 ids, page paging, and keeps unique works.
async function backward(IDS,R){for(let i=0;i<IDS.length;i+=40){const g=IDS.slice(i,i+40);let page=1;
 while(page<=15){const j=await (await fetch('https://api.openalex.org/works?filter=cited_by:'+g.join('|')+',from_publication_date:2022-11-01&per-page=200&select=id,doi,title,publication_year,publication_date,type,cited_by_count,primary_location,best_oa_location,open_access,authorships,abstract_inverted_index&page='+page)).json();
  for(const w of j.results){R[w.id]=R[w.id]||{id:w.id,title:w.title};}
  if(page*200>=j.meta.count)break;page++;}}}
// saved as openalex_backward_snowball_round1_2026-11-09.json

In [ ]:
%%writefile scripts/browser_js/05_resolve_arxiv_and_download.js
// Full text retrieval (open access only). 1) For each record, find an arXiv id in its OpenAlex locations (filter openalex:W1|W2..., select=id,locations).
// 2) On https://arxiv.org (same origin), fetch https://arxiv.org/pdf/<id>, check the %PDF header, pause 1.5 to 2.5 s, and pack 30 PDFs per ZIP (store method, CRC32) downloaded to the Downloads folder as <prefix>NN.zip.
// File names: <recordID>_arxiv_<arxivID>.pdf. ACL Anthology PDFs were fetched the same way on https://aclanthology.org; when downloads from that site were blocked, the arXiv version was used.
// Failures (404) are logged and treated as not retrieved (EXC8 rule applies if the abstract is insufficient).

In [ ]:
%%writefile scripts/browser_js/README.md
# Browser scripts (run in my Chrome, 9 Nov 2026)
The OpenAlex, Google Scholar and arXiv endpoints were blocked from the cloud workspace and from the PC shell, so retrieval ran as JavaScript in the reviewer's own Chrome tab (same origin as the API or site), and results were saved as JSON or ZIP downloads, then copied into data/ and search/raw_exports/. These files are the code that was executed (lightly tidied, logic unchanged). Each also has a Python equivalent in the proof notebook.
Files: 01_openalex_search.js, 02_google_scholar.js, 03_forward_snowball.js, 04_backward_snowball.js, 05_resolve_arxiv_and_download.js

In [ ]:
save_checks('02_Search')